<a href="https://colab.research.google.com/github/isparshgirics24-glitch/AI-LAB/blob/main/1BF24CS125_WEEK2_IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A*

In [2]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g  # Cost from start to current node
        self.h = h  # Heuristic cost to goal
        self.f = g + h  # Total estimated cost

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def get_neighbors(self):
        neighbors = []
        r, c = self.get_blank_pos()
        moves = {
            "Up": (r - 1, c),
            "Down": (r + 1, c),
            "Left": (r, c - 1),
            "Right": (r, c + 1)
        }

        for direction, (nr, nc) in moves.items():
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Create a copy of the board and swap values
                new_board = [row[:] for row in self.board]
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(PuzzleState(new_board, self, direction, self.g + 1))
        return neighbors

def manhattan_distance(board, goal):
    distance = 0
    # Map coordinates of the goal board for fast lookup
    goal_coords = {}
    for r in range(3):
        for c in range(3):
            goal_coords[goal[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                target_r, target_c = goal_coords[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def board_to_tuple(board):
    return tuple(tuple(row) for row in board)

def solve_a_star(initial_board, goal_board):
    initial_state = PuzzleState(
        initial_board,
        g=0,
        h=manhattan_distance(initial_board, goal_board)
    )

    open_set = []
    heapq.heappush(open_set, initial_state)
    closed_set = set()

    while open_set:
        current_state = heapq.heappop(open_set)

        if current_state.board == goal_board:
            # Reconstruct path
            path = []
            curr = current_state
            while curr.parent is not None:
                path.append((curr.move, curr.board))
                curr = curr.parent
            path.reverse()
            return path

        closed_set.add(board_to_tuple(current_state.board))

        for neighbor in current_state.get_neighbors():
            if board_to_tuple(neighbor.board) in closed_set:
                continue

            neighbor.h = manhattan_distance(neighbor.board, goal_board)
            neighbor.f = neighbor.g + neighbor.h

            heapq.heappush(open_set, neighbor)

    return None

def get_user_board(prompt_name):
    print(f"Enter the {prompt_name} state (3x3 grid, use 0 for the blank tile).")
    print("Provide 3 space-separated numbers per row:")
    board = []
    for i in range(3):
        while True:
            try:
                row_input = input(f"Row {i+1}: ").strip().split()
                row = [int(x) for x in row_input]
                if len(row) != 3:
                    raise ValueError("Each row must contain exactly 3 numbers.")
                board.append(row)
                break
            except ValueError as e:
                print(f"Invalid input: {e}. Please try again.")
    return board

# Get states from user input
initial_puzzle = get_user_board("Initial")
print()
goal_puzzle = get_user_board("Goal")
print()

solution = solve_a_star(initial_puzzle, goal_puzzle)

if solution:
    print(f"Solved in {len(solution)} steps!")
    print("Initial State:")
    for r in initial_puzzle: print(r)
    print()

    for step, (move, board) in enumerate(solution, 1):
        print(f"Step {step}: Move {move}")
        for r in board: print(r)
        print()
else:
    print("No solution found.")

Enter the Initial state (3x3 grid, use 0 for the blank tile).
Provide 3 space-separated numbers per row:
Row 1: 283
Invalid input: Each row must contain exactly 3 numbers.. Please try again.
Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 7 0 5

Enter the Goal state (3x3 grid, use 0 for the blank tile).
Provide 3 space-separated numbers per row:
Row 1: 1 2 3
Row 2: 8 0 4
Row 3: 7 6 5

Solved in 5 steps!
Initial State:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Step 1: Move Up
[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

Step 2: Move Up
[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

Step 3: Move Left
[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

Step 4: Move Down
[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

Step 5: Move Right
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]



In [3]:
def get_blank_pos(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def get_neighbors(board):
    neighbors = []
    r, c = get_blank_pos(board)
    moves = {
        "Up": (r - 1, c),
        "Down": (r + 1, c),
        "Left": (r, c - 1),
        "Right": (r, c + 1)
    }

    for direction, (nr, nc) in moves.items():
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_board = [row[:] for row in board]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
            neighbors.append((direction, new_board))
    return neighbors

def depth_limited_search(board, goal, depth, path, visited):
    if board == goal:
        return path
    if depth <= 0:
        return None

    board_tuple = tuple(tuple(row) for row in board)
    visited.add(board_tuple)

    for move, neighbor in get_neighbors(board):
        neighbor_tuple = tuple(tuple(row) for row in neighbor)
        if neighbor_tuple not in visited:
            path.append((move, neighbor))
            result = depth_limited_search(neighbor, goal, depth - 1, path, visited)
            if result is not None:
                return result
            path.pop()

    visited.remove(board_tuple)
    return None

def solve_ids(initial_board, goal_board, max_depth=50):
    for depth in range(max_depth):
        visited = set()
        path = []
        result = depth_limited_search(initial_board, goal_board, depth, path, visited)
        if result is not None:
            return result
    return None

def get_user_board(prompt_name):
    print(f"Enter the {prompt_name} state (3x3 grid, use 0 for the blank tile).")
    print("Provide 3 space-separated numbers per row:")
    board = []
    for i in range(3):
        while True:
            try:
                row_input = input(f"Row {i+1}: ").strip().split()
                row = [int(x) for x in row_input]
                if len(row) != 3:
                    raise ValueError("Each row must contain exactly 3 numbers.")
                board.append(row)
                break
            except ValueError as e:
                print(f"Invalid input: {e}. Please try again.")
    return board

# Get states from user input
initial_puzzle = get_user_board("Initial")
print()
goal_puzzle = get_user_board("Goal")
print()

solution = solve_ids(initial_puzzle, goal_puzzle)

if solution is not None:
    print(f"Solved in {len(solution)} steps!")
    print("Initial State:")
    for r in initial_puzzle: print(r)
    print()

    for step, (move, board) in enumerate(solution, 1):
        print(f"Step {step}: Move {move}")
        for r in board: print(r)
        print()
else:
    print("No solution found within depth limit.")

Enter the Initial state (3x3 grid, use 0 for the blank tile).
Provide 3 space-separated numbers per row:
Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 7 0 5

Enter the Goal state (3x3 grid, use 0 for the blank tile).
Provide 3 space-separated numbers per row:
Row 1: 1 2 3
Row 2: 8 0 4
Row 3: 7 6 5

Solved in 5 steps!
Initial State:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Step 1: Move Up
[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

Step 2: Move Up
[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

Step 3: Move Left
[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

Step 4: Move Down
[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

Step 5: Move Right
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

